# Mapas de tempo de acesso — Porto Alegre

## 1. Bibliotecas e caminhos para Porto Alegre


In [1]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import colors as mpl_colors
from matplotlib.patches import Patch

BASE_DIR = Path.cwd().parent.parent
RESULTS_DIR = BASE_DIR / "results" / "porto_alegre"
OUTPUT_DIR = Path.cwd()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("raiz:", BASE_DIR)
print("resultados:", RESULTS_DIR)
print("saída:", OUTPUT_DIR)


raiz: /home/matheus/Documentos/mestrado/dissertacao
resultados: /home/matheus/Documentos/mestrado/dissertacao/results/porto_alegre
saída: /home/matheus/Documentos/mestrado/dissertacao/notebooks/analysis


## 2. Cenários e escala comum de cores

Os quatro cenários de Porto Alegre são os da pipeline. A escala e a paleta são definidas **uma vez**, fora do laço de plotagem — é isso que garante que a cor de um mesmo tempo signifique o mesmo em todos os mapas, e também o mesmo entre Curitiba e Porto Alegre.


In [2]:
SCENARIOS = [
    "typical",
    "inundacao",
    "movimento_massa",
    "combinado",
]

LABELS = {
    "typical": "Típico",
    "inundacao": "Inundação",
    "movimento_massa": "Movimento de massa",
    "combinado": "Combinado",
}


## 3. Carregamento do tempo de acesso por cenário

Cada matriz origem-destino é reduzida a **um valor por célula**: a média do tempo de viagem aos POIs considerados.

Atenção a um detalhe de Porto Alegre: nos cenários de movimento de massa e combinado, uma parte das células tem tempo **infinito** — são células que perdem totalmente o acesso, porque todas as rotas até os POIs passam por uma área interditada. Elas são separadas aqui, não entram na média, e aparecem em cinza no mapa.


In [3]:
grid = gpd.read_parquet(RESULTS_DIR / "prepared" / "grid.parquet")
print("células na malha:", len(grid), "| CRS:", grid.crs.to_epsg())

acesso = {}
inacessiveis = {}

for cen in SCENARIOS:
    matriz = pd.read_parquet(RESULTS_DIR / "matrices" / f"matrix_{cen}.parquet")
    pth = matriz.groupby("cell_idx")["travel_time"].mean()

    # `inf` significa que a célula perdeu o acesso: não existe rota até nenhum
    # POI. Esses valores são retirados da série antes de plotar e guardados à
    # parte, para virar a categoria cinza.
    infinito = ~np.isfinite(pth)
    inacessiveis[cen] = pth.index[infinito]
    acesso[cen] = pth[~infinito]

    print(f"{cen:18s} com valor: {len(acesso[cen]):5d} "
          f"| inacessíveis: {infinito.sum():5d} "
          f"| mediana: {acesso[cen].median():6.2f} min")


células na malha: 5065 | CRS: 4326
typical            com valor:  4358 | inacessíveis:     0 | mediana:  27.27 min
inundacao          com valor:  4358 | inacessíveis:     0 | mediana:  34.82 min
movimento_massa    com valor:  2292 | inacessíveis:  2066 | mediana:  20.00 min
combinado          com valor:  2292 | inacessíveis:  2066 | mediana:  31.59 min


## 4. Definição dos limites de classe compartilhados


In [9]:
todos = pd.concat(acesso.values())
todos = todos[np.isfinite(todos)]

FAIXAS = [0, 10, 15, 30, 50]

CORES = [
    "#1f4e79",   # 0–10   azul forte
    "#7fb3d5",   # 10–15  azul claro
    "#ffe066",   # 15–30  amarelo
    "#ff9f43",   # 30–50  laranja
    "#c0392b",   # 50+    vermelho
]

ROTULOS = [
    "0–10 min",
    "10–15 min",
    "15–30 min",
    "30–50 min",
    "> 50 min",
]

CMAP_MAPA = mpl_colors.ListedColormap(CORES, name="acesso")

LIMITES_NORM = np.array(FAIXAS + [np.inf], dtype=float)
NORM = mpl_colors.BoundaryNorm(LIMITES_NORM, ncolors=len(CORES))

## 5. Função de plotagem individual com escala fixa

Uma única função desenha qualquer cenário na escala compartilhada. A célula cinza representa duas situações distintas, ambas sem tempo calculado: células que perdem o acesso (tempo infinito) e células que não alcançam nenhum POI pela rede.


In [5]:
def plotar(cen: str, ax=None, mostrar_legenda: bool = True, tamanho_titulo: int = 24):
    """Desenha o mapa de tempo de acesso de um cenário na escala compartilhada."""
    celulas = grid.merge(
        acesso[cen].rename("access_time"), left_on="cell_idx", right_index=True, how="left"
    )

    proprio = ax is None
    if proprio:
        fig = plt.figure(figsize=(12, 13))
        ax = fig.add_axes([0.02, 0.16, 0.96, 0.78])

    celulas.plot(
        column="access_time",
        cmap=CMAP_MAPA,
        norm=NORM,
        ax=ax,
        edgecolor="none",
        missing_kwds={"color": "#bdbdbd"},
        legend=False,
    )
    ax.set_title(LABELS[cen], fontsize=tamanho_titulo, pad=14)
    ax.set_axis_off()

    if proprio and mostrar_legenda:
        legenda = fig.legend(
            handles=[Patch(facecolor=cor, edgecolor="#555555", label=rotulo)
                     for cor, rotulo in zip(CORES, ROTULOS)]
            + [Patch(facecolor="#bdbdbd", edgecolor="#555555", label="Inacessível")],
            loc="lower center",
            ncol=len(CORES) + 1,
            frameon=False,
            fontsize=16,
            handlelength=1.8,
            handleheight=1.4,
            columnspacing=1.6,
            bbox_to_anchor=(0.5, 0.04),
            title="Tempo de acesso (min)",
            title_fontsize=18,
        )
        legenda.get_title().set_fontweight("bold")

    if proprio:
        return fig, ax
    return ax


## 6. Geração dos mapas individuais por cenário


In [6]:
for cen in SCENARIOS:
    fig, _ = plotar(cen)
    destino = OUTPUT_DIR / f"porto_alegre_access_time_{cen}.png"
    fig.savefig(destino, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("salvo:", destino.name)


salvo: porto_alegre_access_time_typical.png
salvo: porto_alegre_access_time_inundacao.png
salvo: porto_alegre_access_time_movimento_massa.png
salvo: porto_alegre_access_time_combinado.png


## 7. Painel comparativo dos cenários

Os quatro mapas numa escala única. A mancha de Porto Alegre é mais larga que a de Curitiba (27,7 × 37,6 km contra 21,1 × 33,3 km), então o painel é 2×2 em vez de 2×3 e cada eixo tem proporção própria.


In [7]:
# A mancha urbana mede 27,7 km × 37,6 km. Com 2 colunas, cada eixo precisa
# dessa proporção para preencher a célula sem sobra de branco lateral.
LARGURA_EIXO = 6.4
ALTURA_EIXO = LARGURA_EIXO * (37.6 / 27.7)

fig = plt.figure(figsize=(2 * LARGURA_EIXO + 0.4, 2 * ALTURA_EIXO + 3.2))
grade = fig.add_gridspec(
    2, 2, left=0.005, right=0.995, top=0.910, bottom=0.130,
    hspace=0.10, wspace=0.005,
)

for i, cen in enumerate(SCENARIOS):
    ax = fig.add_subplot(grade[i // 2, i % 2])
    plotar(cen, ax=ax, mostrar_legenda=False, tamanho_titulo=23)
    ax.set_aspect("equal")
    ax.set_anchor("C")

legenda = fig.legend(
    handles=[Patch(facecolor=cor, edgecolor="#555555", label=rotulo)
             for cor, rotulo in zip(CORES, ROTULOS)]
    + [Patch(facecolor="#bdbdbd", edgecolor="#555555", label="Inacessível")],
    loc="lower center",
    ncol=len(CORES) + 1,
    frameon=False,
    fontsize=19,
    handlelength=1.8,
    handleheight=1.4,
    columnspacing=2.0,
    bbox_to_anchor=(0.5, 0.020),
    title="Tempo de acesso (min)",
    title_fontsize=20,
)
legenda.get_title().set_fontweight("bold")

fig.suptitle("Tempo de acesso por cenário — Porto Alegre", fontsize=28, y=0.975)
fig.savefig(OUTPUT_DIR / "porto_alegre_access_time_painel.png", dpi=300, bbox_inches="tight")
plt.close(fig)
print("salvo: porto_alegre_access_time_painel.png")


salvo: porto_alegre_access_time_painel.png
